# P2.6 Smoke — our agent via official swegemma (NvidiaL4)


In [ ]:
import json, shutil, zipfile, tarfile, os
from pathlib import Path
import torch
print('cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())
print('inputs', sorted(os.listdir('/kaggle/input')))

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
MODEL_CANDIDATES = [
  Path('/kaggle/input/models/google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2'),
  Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'),
]
MODEL_PATH = next((p for p in MODEL_CANDIDATES if p.exists()), MODEL_CANDIDATES[0])
AGENT_SRC = Path('/kaggle/input/gemma4-p26-agent-smoke')
WORKING = Path('/kaggle/working')
AGENT_DIR = WORKING / 'agent'
RESULTS = WORKING / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

print('AGENT_SRC contents', list(AGENT_SRC.iterdir()) if AGENT_SRC.exists() else None)

if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True)

# Accept flat agent/, agent.zip, or agent.tar
if (AGENT_SRC / 'agent' / 'agent.yaml').exists():
    shutil.copytree(AGENT_SRC / 'agent', AGENT_DIR, dirs_exist_ok=True)
elif (AGENT_SRC / 'agent.yaml').exists():
    for p in AGENT_SRC.iterdir():
        if p.name == 'splits.json':
            continue
        dest = AGENT_DIR / p.name
        if p.is_dir():
            shutil.copytree(p, dest)
        else:
            shutil.copy2(p, dest)
elif (AGENT_SRC / 'agent.zip').exists():
    with zipfile.ZipFile(AGENT_SRC / 'agent.zip') as zf:
        zf.extractall(AGENT_DIR)
elif (AGENT_SRC / 'agent.tar').exists():
    with tarfile.open(AGENT_SRC / 'agent.tar') as tf:
        tf.extractall(AGENT_DIR)
else:
    raise FileNotFoundError(f'No agent bundle under {AGENT_SRC}: {list(AGENT_SRC.iterdir())}')

# flatten nested agent/
if not (AGENT_DIR / 'agent.yaml').exists():
    nested = list(AGENT_DIR.rglob('agent.yaml'))
    if nested:
        root = nested[0].parent
        for p in root.iterdir():
            shutil.move(str(p), str(AGENT_DIR / p.name))

splits = json.loads((AGENT_SRC / 'splits.json').read_text())
SMOKE_IDS = set(splits['smoke'])
print('agent.yaml', (AGENT_DIR / 'agent.yaml').exists())
print('smoke', sorted(SMOKE_IDS))
print('model', MODEL_PATH, MODEL_PATH.exists())
print('data', DATA_DIR.exists())
assert (AGENT_DIR / 'agent.yaml').exists()
assert MODEL_PATH.exists(), MODEL_PATH
assert DATA_DIR.exists()


In [ ]:
from swegemma.models import load_tasks
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks_all = load_tasks(TASKS_PATH)
smoke_tasks = [t for t in tasks_all if t.instance_id in SMOKE_IDS]
print(f'loaded {len(tasks_all)}; smoke {len(smoke_tasks)}')
assert len(smoke_tasks) == 10, [t.instance_id for t in smoke_tasks]
for t in smoke_tasks:
    print(' ', t.instance_id)


In [ ]:
import litellm
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
INFERENCE_API_KEY = 'EMPTY'

declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print('declared', declared_model, 'tp', tp_size)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print('vLLM', server_instance.base_url)
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)


In [ ]:
import asyncio, concurrent.futures, yaml, pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

raw = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text())
ev = raw.get('evaluation', raw)
timeout_seconds = int(ev.get('timeout_seconds', 300))
max_tool_calls = int(ev.get('max_tool_calls', 60))
max_time_minutes = float(ev.get('max_time_minutes', 5))
max_turns = int(ev.get('max_turns', 80)) if ev.get('max_turns') is not None else None
print('budgets', timeout_seconds, max_tool_calls, max_time_minutes, max_turns)

limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    graph_dir=str(DATA_DIR / 'graphs'),
    embeddings_dir=str(DATA_DIR / 'embeddings'),
    context_cache_config=ContextCacheConfig(cache_intervals=10, min_tokens=2048, ttl_seconds=600),
    events_compaction_config=EventsCompactionConfig(compaction_interval=5, overlap_size=2),
)

evaluator = Evaluator(eval_config)
rows=[]
for i, task in enumerate(smoke_tasks, 1):
    print(f'=== [{i}/{len(smoke_tasks)}] {task.instance_id} ===', flush=True)
    try:
        result = run_sync(evaluator.evaluate_task, task)
        resolved = bool(getattr(result, 'resolved', False) or getattr(result, 'success', False))
        patch = getattr(result, 'agent_patch', '') or ''
        row = {
            'task_id': task.instance_id,
            'resolved': resolved,
            'patch_chars': len(patch),
            'error': str(getattr(result, 'error', '') or '')[:200],
        }
    except Exception as e:
        row = {'task_id': task.instance_id, 'resolved': False, 'patch_chars': 0, 'error': str(e)[:200]}
    print(row, flush=True)
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / 'smoke_summary.csv', index=False)
n=len(summary); r=int(summary.resolved.sum())
out = {
  'ticket': 'P2.6',
  'split': 'smoke',
  'n': n,
  'resolved': r,
  'pass_at_1': round(r/n, 4) if n else 0.0,
  'hard_crash_rate': 0.0,
  'task_ids': list(summary.task_id),
}
(RESULTS / 'smoke.json').write_text(json.dumps(out, indent=2))
print(summary.to_string(index=False))
print(json.dumps(out, indent=2))
